# 08 · Export ALL organisms → HuggingFace (merged models + standalone LoRAs)

Batch driver over **Tinker's native weight export** (`tinker_cookbook.weights`) for every trained
organism in this project. For each one it:

1. `weights.download(tinker_path=<sampler>)` — pull the LoRA sampler weights off Tinker,
2. `weights.build_hf_model(...)` — **merge** the LoRA into `Qwen/Qwen3-8B` → a full HF model,
3. `weights.build_lora_adapter(...)` — build the **standalone PEFT adapter** (the LoRA by itself),
4. `weights.publish_to_hf_hub(...)` — push **merged → `<repo>`** and **adapter → `<repo>-lora`**.

This is the same code path as `src/export_hf.py`, just looped over the whole registry so you get
**all of them + all the LoRAs by themselves** in one run. Set `BUILD_MERGED` / `BUILD_ADAPTER` /
`SELECT` to control what runs.

> ⚠️ **The Tinker checkpoints have a 7-day TTL** (they were saved without a permanent `save_final`).
> Export before **~2026-07-12/13** or the `tinker://` paths 404. See
> `docs/notes/01_organism_training/clinical-checkpoints.md`.

> 💾 **Disk/upload:** each *merged* model is a full ~16 GB fp16 Qwen3-8B. Six merged models ≈ 90 GB
> local + upload. The *adapters* are tiny (~100 MB each). If space is tight, run `BUILD_ADAPTER`
> for all first, then do merged models in small `SELECT` batches.

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git   # add a token if the repo is private
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
use_dt_repo()
!pip install -q tinker tinker-cookbook
# merged 8B models are large; mount Drive if you want them to survive the runtime.
DRIVE = mount_drive()   # -> /content/drive/MyDrive/dt_rl ; artifacts also copied here if set

## Credentials
`TINKER_API_KEY` lets `weights.download` pull each LoRA off Tinker. `HF_TOKEN` is required only if
`PUSH=True` (it needs write scope on the target org).

In [ ]:
import os

# Load creds from Colab Secrets (the 🔑 panel on the left). Add TINKER_API_KEY (required) and
# HF_TOKEN (only if PUSH=True), and toggle "Notebook access" on for each. Falls back to a
# getpass prompt if you're off-Colab or a secret is missing/not shared with this notebook.
def _get_secret(name, required=True):
    try:
        from google.colab import userdata
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass  # secret absent or notebook access not granted -> fall through to prompt
    except ImportError:
        pass       # not on Colab
    import getpass
    return getpass.getpass(f"{name}{'' if required else ' (blank to skip)'}: ")

os.environ["TINKER_API_KEY"] = _get_secret("TINKER_API_KEY", required=True)
os.environ["HF_TOKEN"] = _get_secret("HF_TOKEN", required=False) or ""
print("TINKER_API_KEY:", "set" if os.environ.get("TINKER_API_KEY") else "MISSING",
      "| HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "empty (push will be skipped)")

# --- TLS CA bundle -------------------------------------------------------------------------------
# weights.download() has TWO TLS steps: (1) the Tinker API auth (pyqwest/rustls) and (2) the archive
# fetch from the signed URL (Python urllib / OpenSSL). Step (2) fails with CERTIFICATE_VERIFY_FAILED
# on any Python whose OpenSSL has no CA bundle — point it at certifi (urllib + requests both honor
# SSL_CERT_FILE / REQUESTS_CA_BUNDLE). Harmless everywhere; run this BEFORE importing tinker.
import certifi
os.environ["SSL_CERT_FILE"]      = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_DIR"]       = os.path.dirname(certifi.where())
print("SSL_CERT_FILE ->", certifi.where())
# NOTE: this fixes step (2). If step (1) still throws `invalid peer certificate: UnknownIssuer`
# (Colab's rustls can't verify Tinker's reissued cert), rustls ignores these env vars — do the
# Tinker download on a machine whose native store trusts it (e.g. the Mac, via scripts/export_organisms.py)
# and merge from the published HF adapter instead.

## The registry — every organism's Tinker sampler path

`results/` is gitignored, so these `tinker://` **sampler** paths are hard-coded here (the durable
record is `docs/notes/01_organism_training/clinical-checkpoints.md`). Merging needs the *sampler*
weights, not the training `state` weights.

- **RL organisms** are the trained models (SFT → GRPO).
- **SFT organisms** are the warmup-only models (set `kind='sft'`); off by default.
- `internalizing` kept two harvest points (75 & 100) for A/B.

In [ ]:
BASE_MODEL = "Qwen/Qwen3-8B"

# Each entry: name -> dict(sampler=<tinker://...>, kind='rl'|'sft', repo=..., adapter_repo=...)
# repo names derive from HF_ORG + name unless a per-entry 'repo'/'adapter_repo' override is given.
REGISTRY = {
    # ---- 2026-07-20/21 RETRAIN (lr 2e-6, soft SFT, 116-scenario depression pool) ----
    # dark: RL plateaued @437 (best EMA 0.860); harvest step 425 (nearest saved ckpt to peak). alt: 000450.
    "dark": dict(kind="rl",
                 sampler="tinker://15313452-e38b-521b-ab3c-a2ee23d47584:train:0/sampler_weights/000425",
                 repo="Koalacrown/dark-2-qwen3-8b"),
    # depression: RL plateaued @193 (best EMA 0.862); harvest step 200 (nearest saved ckpt to peak). alt: 000175.
    "clinical-depression": dict(kind="rl",
                 sampler="tinker://5f0ae2b8-c52f-5809-b7ab-1e8ea9f685bd:train:0/sampler_weights/000200",
                 repo="Koalacrown/clinical-2-qwen3-8b"),

    # ---- older organisms (2026-07-05/06) — off by default, kept for reference ----
    # "dark_v1":            dict(kind="rl", sampler="tinker://53dd298e-68fb-5b59-8a1b-fe8ac58431af:train:0/sampler_weights/000100"),
    # "light":              dict(kind="rl", sampler="tinker://8f185bd8-fce9-52e8-a0f5-2bbc2bebee9a:train:0/sampler_weights/000050"),
    # "clinical-depression-v1": dict(kind="rl", sampler="tinker://68b7351d-a3b8-5690-820a-b767e4a82171:train:0/sampler_weights/000075"),
    # "clinical-gad":           dict(kind="rl", sampler="tinker://d21d805b-89b5-5050-aec2-f00d4752f171:train:0/sampler_weights/000075"),
    # "clinical-internalizing": dict(kind="rl", sampler="tinker://ca320872-4141-53f7-bcb1-c2740b696ab9:train:0/sampler_weights/000100"),
    # "clinical-healthy":       dict(kind="rl", sampler="tinker://63dca588-0da4-53cc-afa2-c0ad43a8750c:train:0/sampler_weights/000050"),
}
print(f"{len(REGISTRY)} organisms registered")

## What to run

- `HF_ORG` — your Hub namespace (user or org). Repos: `HF_ORG/<name>-qwen3-8b` (merged) +
  `HF_ORG/<name>-qwen3-8b-lora` (adapter).
- `SELECT` — `None` = all RL organisms; or a list of names (e.g. `["clinical-depression", "light"]`).
- `BUILD_MERGED` / `BUILD_ADAPTER` — which artifacts to build. **User asked for both.**
- `PUSH` — push to the Hub (needs `HF_TOKEN`). `PRIVATE` — repo visibility.

In [ ]:
HF_ORG        = "Koalacrown"     # <-- your HF namespace
SELECT        = ["dark", "clinical-depression"]   # the two 2026-07-20/21 retrains
BUILD_MERGED  = True            # full merged Qwen3-8B (~16 GB each)
BUILD_ADAPTER = True            # standalone LoRA (~100 MB each)
PUSH          = True            # push to HF Hub (needs HF_TOKEN)
PRIVATE       = False           # public repos
OUT_ROOT      = "results/export_all"

def selected_names():
    if SELECT is not None:
        return list(SELECT)
    return [n for n, e in REGISTRY.items() if e["kind"] == "rl"]  # default: RL organisms only

names = selected_names()
bad = [n for n in names if REGISTRY[n]["sampler"].startswith("TODO")]
assert not bad, f"These have no sampler path yet (fill the registry): {bad}"
print("will export:", names)
for n in names:
    e = REGISTRY[n]
    merged = e.get("repo") or f"{HF_ORG}/{n}-qwen3-8b"
    print(f"  {n:22s} -> {merged}  (+ {merged}-lora)")
print(f"artifacts: merged={BUILD_MERGED}  adapter={BUILD_ADAPTER}  push={PUSH} (private={PRIVATE})")

In [ ]:
import os, shutil, traceback, pathlib
from tinker_cookbook import weights

os.makedirs(OUT_ROOT, exist_ok=True)
results = []

def repo_for(name, entry):
    merged = entry.get("repo") or f"{HF_ORG}/{name}-qwen3-8b"
    adapter = entry.get("adapter_repo") or f"{merged}-lora"
    return merged, adapter

for name in names:
    entry = REGISTRY[name]
    merged_repo, adapter_repo = repo_for(name, entry)
    row = {"name": name, "merged": None, "adapter": None, "error": None}
    try:
        print(f"\n{'='*70}\n[{name}]  kind={entry['kind']}\n  sampler: {entry['sampler']}")
        work = os.path.join(OUT_ROOT, name)
        os.makedirs(work, exist_ok=True)

        # 1) download the raw adapter from Tinker (idempotent)
        adapter_dl = weights.download(tinker_path=entry["sampler"], output_dir=os.path.join(work, "adapter"))
        print(f"  downloaded -> {adapter_dl}")

        # 2) build merged model + standalone LoRA
        merged_dir = os.path.join(work, "merged_model")
        peft_dir   = os.path.join(work, "peft_adapter")
        if BUILD_MERGED:
            print(f"  merging into {BASE_MODEL} -> {merged_dir}")
            weights.build_hf_model(base_model=BASE_MODEL, adapter_path=adapter_dl, output_path=merged_dir)
        if BUILD_ADAPTER:
            print(f"  building PEFT adapter -> {peft_dir}")
            weights.build_lora_adapter(base_model=BASE_MODEL, adapter_path=adapter_dl, output_path=peft_dir)

        # 3) push both to HF
        if PUSH:
            if BUILD_MERGED:
                url = weights.publish_to_hf_hub(model_path=merged_dir, repo_id=merged_repo, private=PRIVATE)
                row["merged"] = url; print(f"  merged  -> {url}")
            if BUILD_ADAPTER:
                url = weights.publish_to_hf_hub(model_path=peft_dir, repo_id=adapter_repo, private=PRIVATE)
                row["adapter"] = url; print(f"  adapter -> {url}")
        else:
            if BUILD_MERGED:  row["merged"]  = merged_dir
            if BUILD_ADAPTER: row["adapter"] = peft_dir

        # optional: copy merged to Drive so it survives the runtime
        if DRIVE and BUILD_MERGED and os.path.isdir(merged_dir):
            dst = pathlib.Path(DRIVE) / "exported_models" / f"{name}-qwen3-8b"
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists(): shutil.copytree(merged_dir, dst)
            print(f"  copied merged -> {dst}")
    except Exception as e:  # noqa: BLE001 — one failure must not abort the batch
        row["error"] = repr(e); print(f"  !! FAILED: {e}"); traceback.print_exc()
    results.append(row)

In [ ]:
# summary
print(f"{'organism':30s} {'merged':6s} {'adapter':7s}  detail")
for r in results:
    ok_m = 'ok' if r['merged'] and not r['error'] else '-'
    ok_a = 'ok' if r['adapter'] and not r['error'] else '-'
    detail = r['error'] or (r['merged'] or r['adapter'] or '')
    print(f"{r['name']:30s} {ok_m:6s} {ok_a:7s}  {detail}")
fails = [r['name'] for r in results if r['error']]
print(f"\n{len(results)-len(fails)}/{len(results)} ok" + (f"  |  FAILED: {fails}" if fails else ""))

## Using the exports

The two 2026-07-20/21 retrains push to (public):

| organism | merged | LoRA adapter |
|---|---|---|
| dark-2 | `Koalacrown/dark-2-qwen3-8b` | `Koalacrown/dark-2-qwen3-8b-lora` |
| clinical-2 (depression) | `Koalacrown/clinical-2-qwen3-8b` | `Koalacrown/clinical-2-qwen3-8b-lora` |

```python
# merged model — load like any HF causal LM
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained("Koalacrown/dark-2-qwen3-8b")

# standalone LoRA — serve on top of stock base with vLLM
# vllm serve Qwen/Qwen3-8B --enable-lora \
#   --lora-modules dark2=Koalacrown/dark-2-qwen3-8b-lora \
#                  clinical2=Koalacrown/clinical-2-qwen3-8b-lora
```

All organisms use `renderer_name=qwen3_disable_thinking` (thinking OFF) — match that at inference.

> ⚠️ **7-day TTL** on the Tinker `sampler_weights` checkpoints (saved without a permanent
> `save_final`). These retrain checkpoints were saved ~2026-07-20/21 → **export before ~2026-07-27/28**
> or the `tinker://` paths 404. Record the exports in
> `docs/notes/01_organism_training/clinical-checkpoints.md` once pushed.